In [3]:
# code1
import sys
import os

# Agrega el path raíz del proyecto
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

from data.data_reader_2023 import *  # ahora funcionará

project_root: str = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("Project root:", project_root)


Project root: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProjectBase


In [2]:
#code 2
from typing import List, Tuple, Literal, Dict, Any
import os
import random
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import scipy.io as sio
from scipy import fftpack
from scipy.signal import iirnotch, lfilter
from scipy.fft import rfft, rfftfreq


In [4]:
# TYPE: CODE 3

%load_ext autoreload
%autoreload 2
# Recarga automáticamente todos los módulos importados.


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# BLOCK 1 — TYPE: MARKDOWN

## Define Constants and Key Variables


In [5]:
# BLOCK 4 — TYPE: CODE

########### Constants

# Define bandpass filter constants
lowcut: float = 0.5
highcut: float = 120.0
fs: int = 1024
resampleFS: int = 250

# Cada filtro devuelve dos arrays (b, a) de coeficientes
notch_1_b: np.ndarray
notch_1_a: np.ndarray
notch_1_b, notch_1_a = iirnotch(1.0, Q=30.0, fs=resampleFS)

notch_60_b: np.ndarray
notch_60_a: np.ndarray
notch_60_b, notch_60_a = iirnotch(60.0, Q=30.0, fs=resampleFS)

# notch_100_b, notch_100_a = iirnotch(100, Q=30, fs=250)

# Define segment interval length in sec
segment_interval: int = 4
print("Segment Interval:", segment_interval)

# Seizure types
# Si queremos hacer un clasificador no binario, entonces tenemos que redefinir
# los labels que están en el paso 8
binary_classifier_flag: bool = True

if binary_classifier_flag:
    seizure_types: List[str] = ["bckg", "seizure"]
    seizure_session_downsampling_ratio: List[float] = [1.0, 1.0]
    seizure_overlapping_ratio: List[float] = [0.0, 0.75]
else:
    seizure_types: List[str] = ["fnsz", "gnsz", "cpsz", "bckg"]
    seizure_session_downsampling_ratio: List[float] = [1.0, 1.0, 1.0, 1.0]
    seizure_overlapping_ratio: List[float] = [0.75, 0.75, 0.75, 0.0]


Segment Interval: 4


In [6]:
# BLOCK 5 — TYPE: CODE

########## Data path

train_val_root: str = os.path.join(
    project_root,
    "dataset",
    "tuh_eeg_seizure",
    "v2.0.3",
    "edf",
    "train",
)

test_root: str = os.path.join(
    project_root,
    "dataset",
    "tuh_eeg_seizure",
    "v2.0.3",
    "edf",
    "dev",
)

# test_root: str = os.path.join('/datadrive', 'TUSZ_2023', 'edf', 'eval')

print("Train/Val root:", train_val_root)
print("Test root:", test_root)

if binary_classifier_flag:
    save_root: str = os.path.join(
        project_root,
        "dataset",
        "tuh_eeg_seizure",
        "v2.0.3",
        "TUSZ_processed_binary_individual_segments",
    )
else:
    save_root: str = os.path.join(
        project_root,
        "dataset",
        "tuh_eeg_seizure",
        "v2.0.3",
        "TUSZ_processed_multiclass_individual_segments",
    )

print("Save root:", save_root)

if not os.path.exists(save_root):
    os.makedirs(save_root, exist_ok=True)

# Modos de datos permitidos
DataMode = Literal["small", "large", "tiny", "full"]
data_mode: DataMode = "tiny"


Train/Val root: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/train
Test root: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/dev
Save root: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/TUSZ_processed_binary_individual_segments


# BLOCK 2 — TYPE: MARKDOWN

## To delete previous data repo when running a new experiment


In [7]:
# BLOCK 6 — TYPE: CODE

# To delete previous data repo when running a new experiment
# This is to avoid concatenating duplicate data to the existing .npy files

# Construye el path del directorio para este experimento
segment_folder: str = os.path.join(
    save_root,
    f"segment_interval_{segment_interval}_sec",
)
print("Segment folder:", segment_folder)

# Si no existe, lo creamos (experimento nuevo)
if not os.path.exists(segment_folder):
    print("Creating new segment folder:", segment_folder)
    os.makedirs(segment_folder, exist_ok=True)

else:
    # Si ya existe, borramos todo su contenido para evitar duplicados
    # al concatenar archivos .npy en ejecuciones sucesivas
    print("Deleting existing segment folder:", segment_folder)

    filenames: List[str] = os.listdir(segment_folder)

    for filename in filenames:
        file_path: str = os.path.join(segment_folder, filename)

        try:
            # Si es archivo o enlace simbólico, lo eliminamos
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)

            # Si es un directorio, lo borramos recursivamente
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)

        except Exception as e:
            # Captura cualquier error en la eliminación y lo informa
            error_msg: str = f"Failed to delete {file_path}. Reason: {e}"
            print(error_msg)


Segment folder: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/TUSZ_processed_binary_individual_segments/segment_interval_4_sec
Creating new segment folder: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/TUSZ_processed_binary_individual_segments/segment_interval_4_sec


In [8]:
# BLOCK 7 — TYPE: CODE

import time

# Obtener rutas de sesión, lista de pacientes y conteo de tipos de referencia
train_val_paths: List[str]
train_val_patients: List[str]
train_val_reference_type_count: Dict[str, int]

print("Getting all TUSZ 2023 session paths of train_val_root...")
init_start_time: float = time.time()
train_val_paths, train_val_patients, train_val_reference_type_count = get_all_TUSZ_2023_session_paths(
    train_val_root
)
init_end_time: float = time.time()

print(
    "get_all_TUSZ_2023_session_paths(train_val_root) time:",
    init_end_time - init_start_time,
    "seconds",
)
print("\tTotal sessions:", len(train_val_paths))
print("\tTrain and val patients:", len(train_val_patients))
print("\tTrain and val reference type count:", train_val_reference_type_count)

# val_paths, val_patients = get_all_TUSZ_2023_session_paths(val_root)
# print('Total sessions: ', len(val_paths))
# print('Total patients: ', len(val_patients))

print("\nGetting all TUSZ 2023 session paths of test_root...")

# Rutas y pacientes de test
test_paths: List[str]
test_patients: List[str]
test_reference_type_count: Dict[str, int]

init_start_time = time.time()
test_paths, test_patients, test_reference_type_count = get_all_TUSZ_2023_session_paths(
    test_root
)
init_end_time = time.time()

print(
    "get_all_TUSZ_2023_session_paths(test_root) time:",
    init_end_time - init_start_time,
    "seconds",
)
print("\tTest sessions:", len(test_paths))
print("\tTest patients:", len(test_patients))
print("\tTest reference type count:", test_reference_type_count)

# Unión de todas las rutas de sesión
all_paths: List[str] = train_val_paths + test_paths
print("\nAll sessions:", len(all_paths))

# Estructuras para almacenar datos por tipo de clase
# Cada sublista corresponderá a uno de los labels en seizure_types
training_data: List[List[Any]] = [[] for _ in seizure_types]
validation_data: List[List[Any]] = [[] for _ in seizure_types]
testing_data: List[List[Any]] = [[] for _ in seizure_types]

# training_data = [[] for i in range(len(seizure_types))]
# validation_data = [[] for i in range(len(seizure_types))]
# testing_data = [[] for i in range(len(seizure_types))]

print("\nlen training_data:", len(training_data))
print("training_data:", training_data)
print("len validation_data:", len(validation_data))
print("validation_data:", validation_data)
print("len testing_data:", len(testing_data))
print("testing_data:", testing_data)

# Fijamos semilla para reproducibilidad y mezclamos pacientes
random.seed(42)
random.shuffle(train_val_patients)

# División 80/20 de pacientes entre entrenamiento y validación
train_patients: List[str] = train_val_patients[: int(len(train_val_patients) * 0.8)]
val_patients: List[str] = train_val_patients[int(len(train_val_patients) * 0.8) :]

print("---------------------------------------")
print("Train patients:", len(train_patients))
print("Val patients:", len(val_patients))

# Total sessions: 4664
# Train and val patients: 579
# Test sessions: 1832
# Test patients: 53
# All sessions: 6496

print("some .edf files")
train_val_paths[:5]  # Mostrar las primeras 5 rutas de sesión para verificar


Getting all TUSZ 2023 session paths of train_val_root...
get_all_TUSZ_2023_session_paths(train_val_root) time: 0.06437253952026367 seconds
	Total sessions: 4667
	Train and val patients: 579
	Train and val reference type count: {'01_tcp_ar': 683, '02_tcp_le': 324, '03_tcp_ar_a': 168}

Getting all TUSZ 2023 session paths of test_root...
get_all_TUSZ_2023_session_paths(test_root) time: 0.017303466796875 seconds
	Test sessions: 1832
	Test patients: 53
	Test reference type count: {'01_tcp_ar': 268, '02_tcp_le': 38, '03_tcp_ar_a': 36}

All sessions: 6499

len training_data: 2
training_data: [[], []]
len validation_data: 2
validation_data: [[], []]
len testing_data: 2
testing_data: [[], []]
---------------------------------------
Train patients: 463
Val patients: 116
some .edf files


['/home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaooo/s004_2014/01_tcp_ar/aaaaaooo_s004_t003.edf',
 '/home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaooo/s004_2014/01_tcp_ar/aaaaaooo_s004_t000.edf',
 '/home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaooo/s004_2014/01_tcp_ar/aaaaaooo_s004_t002.edf',
 '/home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaooo/s004_2014/01_tcp_ar/aaaaaooo_s004_t001.edf',
 '/home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaooo/s001_2013/01_tcp_ar/aaaaaooo_s001_t000.edf']

In [9]:
# BLOCK 8 — TYPE: CODE

# Probar lectura de un archivo CSV
data_path: str = (
    project_root
    + "/dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000.edf"
)

print("Data path:", data_path)

label_file: str = data_path[:-4] + ".csv"

print("data_path[:-4]:", data_path[:-4])
print("label_file:", label_file)

# Los .csv del dataset TUSZ 2023 tienen un encabezado de 5 filas
df = pd.read_csv(label_file, skiprows=5, header=0)
df


Data path: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000.edf
data_path[:-4]: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000
label_file: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000.csv


,channel,start_time,stop_time,label,confidence
0,FP1-F7,0.0,1205.0,bckg,1.0
1,F7-T3,0.0,1205.0,bckg,1.0
2,T3-T5,0.0,1205.0,bckg,1.0
3,T5-O1,0.0,1205.0,bckg,1.0
4,FP2-F8,0.0,1205.0,bckg,1.0
5,F8-T4,0.0,1205.0,bckg,1.0
6,T4-T6,0.0,1205.0,bckg,1.0
7,T6-O2,0.0,1205.0,bckg,1.0
8,A1-T3,0.0,1205.0,bckg,1.0
9,T3-C3,0.0,1205.0,bckg,1.0


In [10]:
# BLOCK 9 — TYPE: CODE

russellTestPaths: List[str] = [
    # "dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000.edf",
    "dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf",
    # "dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t001.edf",
    # "dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t002.edf",
]

for data_path in russellTestPaths:
    print(data_path)

    dps = data_path.split("train/")
    print("dps:", dps)

    patient: str = data_path.split("train/")[1].split("/")[0]
    print("\tPatient:", patient)

    reference_type: str = data_path.split("train/")[1].split("/")[2]
    print("\tReference type:", reference_type)

    # temp = data_path.split('train/')[1].split('/')
    # print("\tTemp:", temp)

    patient_session: str = data_path.split("train/")[1].split("/")[-1][:-4]
    print("patient_session:", patient_session)


dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf
dps: ['dataset/tuh_eeg_seizure/v2.0.3/edf/', 'aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf']
	Patient: aaaaaauj
	Reference type: 01_tcp_ar
patient_session: aaaaaauj_s004_t000


In [16]:
# BLOCK 10 — TYPE: CODE

from typing import Set

selected_patients: Set[str] = set()
total_bckg_segments: int = 0
total_seiz_segments: int = 0

# dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf
# dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaedy/s002_2004/02_tcp_le/aaaaaedy_s002_t001.edf
# dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaedy/s003_2004/02_tcp_le/aaaaaedy_s003_t000.edf

count_session: int = 0

# sample_paths = train_val_paths[0:100]
sample_paths = test_paths[0:100]

# print("Sample paths:", sample_paths)
# for data_path in train_val_paths:

print("Start processing TUSZ 2023 sessions...")

contador_02_tcp_le_03_tcp_ar_a: int = 0

# for data_path in train_val_paths:
for data_path in sample_paths:
    # for data_path in test_paths:

    # patient: str = data_path.split('train/')[1].split('/')[0]
    patient: str = data_path.split("dev/")[1].split("/")[0]

    # reference_type: str = data_path.split('train/')[1].split('/')[2]
    reference_type: str = data_path.split("dev/")[1].split("/")[2]

    if reference_type == "02_tcp_le" or reference_type == "03_tcp_ar_a":
        contador_02_tcp_le_03_tcp_ar_a += 1
        continue

    # patient_session: str = data_path.split('train/')[1].split('/')[-1][:-4]
    patient_session: str = data_path.split("dev/")[1].split("/")[-1][:-4]


    # patient_session = patient+'_'+data_path.split('01_tcp_ar')[1][14:18]
    # print("after if:", reference_type)

    # --------------------------------------------------
    # 3) Set 'flag_train_val_test' train/val/test
    # --------------------------------------------------
    flag_train_val_test: str

    if patient in train_patients:
        # continue
        flag_train_val_test = "train"
    elif patient in val_patients:
        # continue
        flag_train_val_test = "val"
    # nunca entrara aqui porque test_patients no contiene pacientes de train/val
    elif patient in test_patients:
        flag_train_val_test = "test"
    else:
        print("Patient not found in train, val, or test lists:", patient)
        continue

    # if data_path != '/datadrive/yuandaData/edf/train/01_tcp_ar/091/00009104/s011_2014_09_29/00009104_s011_t000.edf':
    #     continue

    ## ME QUEDE AQUI, CONTINUAMOS DESDE AQUI
    count_session += 1
    selected_patients.add(patient)

    # print('Patient is:', patient)
    # print('Patient belongs to:', flag_train_val_test)
    # print('Patient session is:', patient_session)
    # break

    # --------------------------------------------------
    # 4) Carga de la señal cruda EDF
    # --------------------------------------------------
    raw: mne.io.BaseRaw = mne.io.read_raw_edf(
        data_path,
        preload=True,
        verbose="warning",
    )
    # print(raw.info)

    # Usualmente, el sampling frequency es 250 Hz
    thisFS: int = int(raw.info["sfreq"])
    # if thisFS != 250:
    #     continue

    # --------------------------------------------------
    # 5) Extracción de canales
    # --------------------------------------------------
    flag_wrong: bool
    signals: np.ndarray

    flag_wrong, signals = get_channels_from_raw(raw)

    if flag_wrong:
        print(
            "flag_wrong, hubo un error al cargar la diferencia de canales [if flag_wrong]",
            flag_wrong,
        )
        continue

    # make_a_sample_plot_from_array(signals, thisFS)
    # make_a_frequency_plot_from_array(signals, thisFS)
    # continue

    # print("printing signals shape:")
    # print("shape:", signals.shape)        # (20, 1048576)
    # print("shape[0]:", signals.shape[0])  # 20
    # print("shape[1]:", signals.shape[1])  # 1048576

    # Ejemplos previos de shape:
    # (20, 362250)
    # (20, 86000)
    # (20, 312000)
    # (20, 2750)
    # (20, 311250)
    # (20, 98000)
    # (20, 50000)
    # (20, 86000)

    # --------------------------------------------------
    # 6) Filtrado bandpass + notch
    # --------------------------------------------------
    filtered_signals: List[np.ndarray] = []

    for i in range(signals.shape[0]):
        bandpass_filtered_signal: np.ndarray = butter_bandpass_filter(
            signals[i, :],
            lowcut,
            highcut,
            fs,
            order=3,
        )
        filtered_1_signal: np.ndarray = lfilter(
            notch_1_b,
            notch_1_a,
            bandpass_filtered_signal,
        )
        filtered_60_signal: np.ndarray = lfilter(
            notch_60_b,
            notch_60_a,
            filtered_1_signal,
        )
        filtered_signals.append(filtered_60_signal)

    # print("shape filtrado:", filtered_60_signal.shape)
    # each filtered_60_signal is a numpy array of shape (N,)
    # Ejemplos después del filtrado:
    # (20, 362250)
    # (20, 86000)
    # (20, 312000)
    # (20, 2750)
    # (20, 311250)
    # (20, 98000)
    # (20, 50000)
    # (20, 86000)

    # make_a_filtered_plot_for_comparison(signals, filtered_signals, thisFS)
    # plot_signal_in_frequency(signals[0], filtered_signals[0], thisFS)

    # --------------------------------------------------
    # 7) Remuestreo
    # --------------------------------------------------
    # print("thisFS:", thisFS)
    # print("data_path:", data_path)

    resampled_signals: List[np.ndarray]

    if thisFS == resampleFS:
        # resampleFS is 250 Hz defined above
        # print("thisFS == 250, no resampling needed")
        resampled_signals = filtered_signals[:]
    else:
        # print("thisFS != 250, resampling needed")
        # print("before resampling:", filtered_signals[0].shape)
        # before resampling: (98000,) => 400 Hz x 245 s
        resampled_signals = resample_data_in_each_channel(
            filtered_signals,
            thisFS,
            resampleFS,
        )
        # print("after resampling:", resampled_signals[0].shape)
        # after resampling: (61250,) => 250 Hz x 245 s

    # --------------------------------------------------
    # 8) Leer siempre el CSV binario y completar huecos
    # --------------------------------------------------
    # label_csv: str = data_path[:-4] + '.csv'
    # labels: List[Tuple[int, int, str]] = []
    # with open(label_csv, 'r') as f:
    #     lines: List[str] = f.readlines()
    #     for line in lines:
    #         if line.startswith("#") or line.startswith("channel"):
    #             continue
    #         parts: List[str] = line.strip().split(",")
    #         channel: str = parts[0]
    #         start_time: int = int(float(parts[1]))
    #         stop_time: int = int(float(parts[2]))
    #         label: str = parts[3]
    #         labels.append((start_time, stop_time, label))

    # Estamos leyendo el asociado .csv_bi y posprocesamos completando huecos para bckg
    labels: List[Tuple[int, int, str]] = get_labels_complete_from_csv_bi_clasificacion_binaria(
        data_path
    )

    # --------------------------------------------------
    # 9) Segmentación (binaria o multiclase)
    # --------------------------------------------------
    segments: List[List[List[np.ndarray]]]

    if binary_classifier_flag:
        segments = slice_signals_into_binary_segments(
            resampled_signals,  # filtered_signals,
            resampleFS,         # thisFS,
            labels,
            segment_interval,
            seizure_types,
            seizure_overlapping_ratio,
        )
    else:
        segments = slice_signals_into_multiclass_segments(
            resampled_signals,  # filtered_signals,
            resampleFS,         # thisFS,
            labels,
            segment_interval,
            seizure_types,
            seizure_overlapping_ratio,
        )

    # segments[i]: list para cada clase
    # segments[i][j]: lista de ventanas dentro de un intervalo anotado
    # segments[i][j][k]: np.ndarray de forma (22, FS * segment_interval)

    # --------------------------------------------------
    # 10) Guardado de los .npy
    # --------------------------------------------------
    print("Saving for patient:", patient)

    for i in range(len(segments)):
        if segments[i] and segments[i][0]:
            # Clase y al menos un intervalo con datos
            this_array: List[np.ndarray] = []
            this_labels: str = seizure_types[i]

            # Por cada intervalo
            for j in range(len(segments[i])):
                if not segments[i][j]:
                    continue

                # Por cada ventana
                for k in range(len(segments[i][j])):
                    this_array.append(segments[i][j][k])

            folder_base: str = os.path.join(
                save_root,
                f"segment_interval_{segment_interval}_sec",
                flag_train_val_test,
                this_labels,
            )
            os.makedirs(folder_base, exist_ok=True)

            save_file: str = os.path.join(
                folder_base,
                f"{patient_session}.npy",
            )

            if os.path.isfile(save_file):
                print("\tsave_file exists, appending new data...")

                # Si el archivo existe, cargamos los datos existentes
                existing_data: np.ndarray = np.load(
                    save_file,
                    allow_pickle=True,
                )

                # Concatenamos con los datos nuevos
                new_data: np.ndarray = np.concatenate(
                    (existing_data, np.array(this_array)),
                )

                # Guardamos al archivo
                np.save(save_file, new_data)
                print(new_data.shape)

                if this_labels == "bckg":
                    total_bckg_segments += len(new_data)
                    print("len(new_data):", len(new_data))
                elif this_labels == "seizure":
                    total_seiz_segments += len(new_data)
                    print("len(new_data):", len(new_data))
            else:
                print("\tsave_file:", save_file)
                print(np.array(this_array).shape)

                np.save(save_file, np.array(this_array))

                if this_labels == "bckg":
                    total_bckg_segments += len(this_array)
                    print("LEN(this_array):", len(this_array))
                elif this_labels == "seizure":
                    total_seiz_segments += len(this_array)
                    print("LEN(this_array):", len(this_array))

    # Control por modo de datos
    if data_mode == "small":
        if count_session >= 1500:
            break
    elif data_mode == "tiny":
        if count_session >= 100:  # para eval; para train podría ser 200
            break
    elif data_mode == "large":
        if count_session >= 2500:
            break

print(
    "End processing TUSZ 2023 sessions... with",
    count_session,
    "sessions processed",
)
print("contador_02_tcp_le_03_tcp_ar_a:", contador_02_tcp_le_03_tcp_ar_a)
print("Total sessions processed:", count_session)

print("\n===== RESUMEN DE 'tiny' =====")
print("Pacientes seleccionados:", len(selected_patients))
print("Pacientes IDs:", sorted(selected_patients))
print("Total segmentos bckg :", total_bckg_segments)
print("Total segmentos seizure:", total_seiz_segments)
print("Total sesiones procesadas:", count_session)

Start processing TUSZ 2023 sessions...
Saving for patient: aaaaabxe
	save_file: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/TUSZ_processed_binary_individual_segments/segment_interval_4_sec/test/bckg/aaaaabxe_s002_t002.npy
(135, 22, 1000)
LEN(this_array): 135
Saving for patient: aaaaabxe
	save_file: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/TUSZ_processed_binary_individual_segments/segment_interval_4_sec/test/bckg/aaaaabxe_s002_t001.npy
(606, 22, 1000)
LEN(this_array): 606
	save_file: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure/v2.0.3/TUSZ_processed_binary_individual_segments/segment_interval_4_sec/test/seizure/aaaaabxe_s002_t001.npy
(27, 22, 1000)
LEN(this_array): 27
Saving for patient: aaaaabxe
	save_file: /home/russell/ssd/code/Topicos_Ciencia_Datos/Tesis/EEGNetTransformerProject/dataset/tuh_eeg_seizure